In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
train = pd.read_parquet("../data/interim/features_train.parquet")

In [3]:
test = pd.read_parquet("../data/interim/features_test.parquet")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1296675, 13)
Test shape: (555719, 13)


In [4]:
print(train['trans_date_trans_time'].dtype)

datetime64[us]


In [5]:
#Row count
print("Train:", len(train))
print("Test:", len(test))


Train: 1296675
Test: 555719


In [6]:
#Sort train by time 
train = train.sort_values(
    "trans_date_trans_time"
).reset_index(drop=True)
#Putting transcations in a timeline

In [7]:
#Setting a cut date ( 21 March 2020)
cut_date = pd.to_datetime("2020-03-21 00:00:00")
#This means:
# Everything before this moment → train
# Everything at or after this moment → validation

In [8]:
new_train = train[
    train['trans_date_trans_time'] < cut_date # Strictly before the cut date
].copy()

In [9]:
#Creating a validation 
validation = train[
    train["trans_date_trans_time"] >= cut_date
].copy()
#On or after March 21.

In [10]:
test = test.copy()

In [11]:
print("New train:", len(new_train))
print("Validation:", len(validation))
print("Original train:", len(train))

print(
    "Do they add up?",
    len(new_train) + len(validation) == len(train)
)


New train: 1070966
Validation: 225709
Original train: 1296675
Do they add up? True


In [12]:
print("TRAIN")
print("Start:", new_train["trans_date_trans_time"].min())
print("End:", new_train["trans_date_trans_time"].max())

print("\nVALIDATION")
print("Start:", validation["trans_date_trans_time"].min())
print("End:", validation["trans_date_trans_time"].max())

print("\nTEST")
print("Start:", test["trans_date_trans_time"].min())
print("End:", test["trans_date_trans_time"].max())


TRAIN
Start: 2019-01-01 00:00:18
End: 2020-03-20 23:58:44

VALIDATION
Start: 2020-03-21 00:00:20
End: 2020-06-21 12:13:37

TEST
Start: 2020-06-21 12:14:25
End: 2020-12-31 23:59:34


In [13]:
#Calculate fraud rate
new_train["is_fraud"].mean()*100

np.float64(0.5835852865543817)

In [14]:
# final table:
def get_summary(data):
    return {
        "Start date": data["trans_date_trans_time"].min(),
        "End date": data["trans_date_trans_time"].max(),
        "Rows": len(data),
        "Fraud rate": data["is_fraud"].mean() * 100
    }

In [15]:
summary = pd.DataFrame({
    "Train": get_summary(new_train),
    "Validation": get_summary(validation),
    "Test": get_summary(test)
}).T

print(summary)


                     Start date             End date     Rows Fraud rate
Train       2019-01-01 00:00:18  2020-03-20 23:58:44  1070966   0.583585
Validation  2020-03-21 00:00:20  2020-06-21 12:13:37   225709   0.556469
Test        2020-06-21 12:14:25  2020-12-31 23:59:34   555719   0.385986


In [16]:
summary["Fraud rate"] = summary["Fraud rate"].round(3)


In [17]:
summary["Fraud rate"]

Train         0.584
Validation    0.556
Test          0.386
Name: Fraud rate, dtype: object

## Save the three sets

**Cut date: `2020-03-21`** (train = before, validation = on or after). This later moves into `params.yaml`, so it's never hidden inside code.

The split is saved once, and every later step reads these files. That way every notebook uses exactly the same sets, and nobody redoes the cut with a different date or forgets to sort.

- Input: `data/interim/features_train.parquet`, `features_test.parquet` (from 02)
- Output: `data/processed/train.parquet`, `val.parquet`, `test.parquet`

In [18]:
new_train.to_parquet("../data/processed/train.parquet", index=False)
validation.to_parquet("../data/processed/val.parquet", index=False)
test.to_parquet("../data/processed/test.parquet", index=False)

In [19]:
# Re-open the saved files to prove they match the table above
for name in ["train", "val", "test"]:
    saved = pd.read_parquet(f"../data/processed/{name}.parquet")
    print(f"{name}: {len(saved):,} rows, fraud rate {saved['is_fraud'].mean() * 100:.3f}%")

train: 1,070,966 rows, fraud rate 0.584%
val: 225,709 rows, fraud rate 0.556%
test: 555,719 rows, fraud rate 0.386%


Model prep

In [20]:
train_lr = new_train.copy()
val_lr = validation.copy()
test_lr = test.copy()

In [21]:
median_secs = train_lr["secs_since_last"].median()
median_avg_amt =train_lr["card_avg_amt_before"].median()
median_ratio = train_lr["amt_ratio"].median()

In [22]:
print("secs_since_last median:", median_secs)
print("card_avg_amt_before median:", median_avg_amt)
print("amt_ratio median:", median_ratio)


secs_since_last median: 16469.0
card_avg_amt_before median: 65.02139183055975
amt_ratio median: 0.6659300276774072


In [23]:
train_lr["secs_since_last"] = train_lr["secs_since_last"].fillna(median_secs)
train_lr["card_avg_amt_before"] = train_lr["card_avg_amt_before"].fillna(median_avg_amt)
train_lr["amt_ratio"] = train_lr["amt_ratio"].fillna(median_ratio)


In [24]:
val_lr["secs_since_last"] = val_lr["secs_since_last"].fillna(median_secs)
val_lr["card_avg_amt_before"] = val_lr["card_avg_amt_before"].fillna(median_avg_amt)
val_lr["amt_ratio"] = val_lr["amt_ratio"].fillna(median_ratio)


In [25]:
test_lr["secs_since_last"] = test_lr["secs_since_last"].fillna(median_secs)
test_lr["card_avg_amt_before"] = test_lr["card_avg_amt_before"].fillna(median_avg_amt)
test_lr["amt_ratio"] = test_lr["amt_ratio"].fillna(median_ratio)


In [26]:
print("Train missing:")
print(train_lr.isna().sum().sum())

print("Validation missing:")
print(val_lr.isna().sum().sum())

print("Test missing:")
print(test_lr.isna().sum().sum())


Train missing:
0
Validation missing:
0
Test missing:
0


One-hot encode category

In [27]:
print(train_lr["category"].unique())


['misc_net', 'grocery_pos', 'entertainment', 'gas_transport', 'misc_pos', ..., 'personal_care', 'health_fitness', 'travel', 'kids_pets', 'home']
Length: 14
Categories (14, str): ['entertainment', 'food_dining', 'gas_transport', 'grocery_net', ..., 'personal_care', 'shopping_net', 'shopping_pos', 'travel']


In [28]:
all_categories = pd.concat([
    train_lr["category"],
    val_lr["category"],
    test_lr["category"]
])


In [29]:
category_values = all_categories.unique()

print(category_values)
print("Number of categories:", len(category_values))


['misc_net', 'grocery_pos', 'entertainment', 'gas_transport', 'misc_pos', ..., 'personal_care', 'health_fitness', 'travel', 'kids_pets', 'home']
Length: 14
Categories (14, str): ['entertainment', 'food_dining', 'gas_transport', 'grocery_net', ..., 'personal_care', 'shopping_net', 'shopping_pos', 'travel']
Number of categories: 14


In [30]:
train_cat = pd.get_dummies(
    train_lr["category"],
    prefix="category"
)

val_cat = pd.get_dummies(
    val_lr["category"],
    prefix="category"
)

test_cat = pd.get_dummies(
    test_lr["category"],
    prefix="category"
)


In [31]:
#Make sure all three have the same columns
#Use the train columns as the standard:

val_cat = val_cat.reindex(
    columns=train_cat.columns,
    fill_value=0
)

test_cat = test_cat.reindex(
    columns=train_cat.columns,
    fill_value=0
)


In [32]:
print(train_cat.columns.tolist())
print(val_cat.columns.tolist())
print(test_cat.columns.tolist())


['category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel']
['category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel']
['category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel']


In [33]:
#Remove Original categoiry 
train_lr = train_lr.drop(columns=["category"])
val_lr = val_lr.drop(columns=["category"])
test_lr = test_lr.drop(columns=["category"])


In [34]:
train_lr = pd.concat(
    [train_lr, train_cat],
    axis=1
)

val_lr = pd.concat(
    [val_lr, val_cat],
    axis=1
)

test_lr = pd.concat(
    [test_lr, test_cat],
    axis=1
)


In [35]:
drop_columns = [
    "trans_date_trans_time",
    "cc_num",
    "gender",
    "is_fraud"
]


In [36]:
#Create Feature tables 
X_train = train_lr.drop(columns=drop_columns)
X_val = val_lr.drop(columns=drop_columns)
X_test = test_lr.drop(columns=drop_columns)


In [37]:
y_train = train_lr["is_fraud"]
y_val = val_lr["is_fraud"]
y_test = test_lr["is_fraud"]


In [38]:
print("Train features:", X_train.shape[1])
print("Validation features:", X_val.shape[1])
print("Test features:", X_test.shape[1])


Train features: 22
Validation features: 22
Test features: 22


In [39]:
print(
    "Train == Validation columns:",
    X_train.columns.tolist() == X_val.columns.tolist()
)

print(
    "Train == Test columns:",
    X_train.columns.tolist() == X_test.columns.tolist()
)


Train == Validation columns: True
Train == Test columns: True


In [40]:
#Skew check 
log_columns =[
    'amt',
    'secs_since_last',
    'card_avg_amt_before',
    'amt_ratio'
]
print("Skew check:")
for col in log_columns:
    print(col,":",train_lr[col].skew())

Skew check:
amt : 41.59149104910224
secs_since_last : 4.329522768749509


card_avg_amt_before : 9.551324340362644
amt_ratio : 57.770762543052555


In [41]:
for col in log_columns:
    train_lr[col] =np.log1p(train_lr[col])
    val_lr[col] =np.log1p(val_lr[col])
    test_lr[col] =np.log1p(test_lr[col])

In [42]:
print("Skewness AFTER log:")

for col in log_columns:
    print(col, ":", train_lr[col].skew())


Skewness AFTER log:
amt : -0.29711183898773524
secs_since_last : -0.7515547138681619
card_avg_amt_before : 0.20955149089302033
amt_ratio : 1.5961327532799852


In [43]:
from sklearn.preprocessing import StandardScaler


In [44]:
scaler = StandardScaler()

In [45]:
drop_columns = [
    "trans_date_trans_time",
    "cc_num",
    "gender",
    "is_fraud"
]

X_train = train_lr.drop(columns=drop_columns)
X_val = val_lr.drop(columns=drop_columns)
X_test = test_lr.drop(columns=drop_columns)


In [46]:
scaler.fit(X_train)



StandardScaler()

In [47]:
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [48]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    X_val_scaled,
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)


In [49]:
print("Train amt:")
print("Mean:", X_train_scaled["amt"].mean())
print("Std:", X_train_scaled["amt"].std())

print("\nValidation amt:")
print("Mean:", X_val_scaled["amt"].mean())
print("Std:", X_val_scaled["amt"].std())


Train amt:
Mean: 2.9468224156061544e-16
Std: 1.0000004668685563

Validation amt:
Mean: 0.001902482285027236
Std: 1.0006178235359926


## Baseline model: logistic regression

- `class_weight="balanced"`: missing a fraud costs about 170x more than a false alarm, so the model can't just say "not fraud" every time.
- On validation we take **fraud scores** (`predict_proba`), not yes/no answers, because ROC-AUC and PR-AUC judge the ranking.
- **Test is not touched.** It's the final exam.

In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score

# class_weight="balanced": frauds count much more than normal transactions
# max_iter=1000: gives the model enough tries to finish learning (avoids the "did not converge" warning)
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [51]:
# Fraud scores (0 to 1) on validation: column 1 of predict_proba = chance of fraud
val_scores = log_reg.predict_proba(X_val_scaled)[:, 1]

# Yes/no answers (50% cutoff), only needed for accuracy
val_preds = log_reg.predict(X_val_scaled)

In [52]:
accuracy = accuracy_score(y_val, val_preds)
roc_auc = roc_auc_score(y_val, val_scores)
pr_auc = average_precision_score(y_val, val_scores)   # PR-AUC
random_pr_auc = y_val.mean()                          # a random guess scores about the fraud rate

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC:  {roc_auc:.4f}")
print(f"PR-AUC:   {pr_auc:.4f}  (random = {random_pr_auc:.4f}, so {pr_auc / random_pr_auc:.0f}x better than random)")

Accuracy: 0.9100
ROC-AUC:  0.9331
PR-AUC:   0.2209  (random = 0.0056, so 40x better than random)


In [53]:
# Each column's weight. All columns are on the same scale, so the weights are comparable.
# + pushes toward fraud, - pushes toward normal. "Biggest" = largest size, ignoring the sign.
weights = pd.Series(log_reg.coef_[0], index=X_train_scaled.columns)
weights = weights.reindex(weights.abs().sort_values(ascending=False).index)

print("Top 3 weights:")
print(weights.head(3))
print("\nAll weights:")
print(weights)

Top 3 weights:
amt_ratio              3.343109
amt                   -3.063143
card_avg_amt_before    1.327507
dtype: float64

All weights:
amt_ratio                  3.343109
amt                       -3.063143
card_avg_amt_before        1.327507
category_gas_transport     0.841081
category_shopping_pos     -0.711485
category_shopping_net     -0.597812
secs_since_last           -0.494176
category_grocery_net       0.428389
hour                       0.418832
is_first_txn               0.370082
category_entertainment    -0.354713
category_grocery_pos       0.341862
category_misc_net         -0.326757
category_personal_care     0.202024
category_kids_pets         0.190567
category_home             -0.168100
txn_count_24h             -0.108582
category_misc_pos          0.078222
category_travel           -0.077634
category_health_fitness    0.067472
category_food_dining       0.048966
age                        0.048892
dtype: float64
